# Experiment 1: Inspect the Sparse Representation

- Đọc raw corpus đã cho ở thư mục `data` và lưu các document trong corpus vào `list`

In [1]:
import gzip
import json

data_path = '../data/c4-train.00000-of-01024-30K.json.gz'

corpus = []

with gzip.open(data_path, 'rt', encoding = 'utf-8') as f:
    for line in f:
        data = json.loads(line)
        corpus.append(data['text'])

- In ra một document bất kỳ và số lượng document sau khi đọc dữ liệu

In [2]:
print(corpus[0])
print('--------')
print(f'Number of documents: {len(corpus)}')

Beginners BBQ Class Taking Place in Missoula!
Do you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class for everyone who wants to get better with their culinary skills.
He will teach you everything you need to know to compete in a KCBS BBQ competition, including techniques, recipes, timelines, meat selection and trimming, plus smoker and fire information.
The cost to be in the class is $35 per person, and for spectators it is free. Included in the cost will be either a t-shirt or apron and you will be tasting samples of each meat that is prepared.
--------
Number of documents: 30000


- Xây dựng pipeline biểu diễn ma trận TF-IDF trên dataset

In [3]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

# Tokenizer (tách từ) và CountVectorizer (vector hóa các document)
vectorizer = CountVectorizer(
    lowercase = True,       # Viết thường các từ
    tokenizer = str.split,  # Tách theo khoảng trắng
    token_pattern = None    # Tắt tính năng tách từ theo biểu thức chính quy
)

# Ma trận đếm từ xuất hiện trong corpus
count_matrix = vectorizer.fit_transform(corpus)
print(f'Count matrix shape: {count_matrix.shape}')

# Trích ra danh sách từ xuất hiện trong corpus
vocabulary_list = vectorizer.get_feature_names_out()
print(f'Vocabulary size: {len(vocabulary_list)}')

# Ma trận TF
document_lengths = np.asarray(count_matrix.sum(axis=1)).ravel()
tf_matrix = count_matrix.multiply(1 / document_lengths[:, None])

# Vector IDF
N = len(corpus)
df_vector = np.asarray((count_matrix > 0).sum(axis=0)).ravel() # Vector DF

idf_vector = np.log(N / df_vector)

# Ma trận TF-IDF
tf_idf_matrix = tf_matrix.multiply(idf_vector)
print(f'TF-IDF matrix shape: {tf_idf_matrix.shape}')

Count matrix shape: (30000, 473388)
Vocabulary size: 473388
TF-IDF matrix shape: (30000, 473388)


- N (số document): 30000
- V (số vocab): 473388

- Tính độ thưa (sparsity) của ma trận TF-IDF

In [4]:
S = 1 - (tf_idf_matrix.nnz / (N * len(vocabulary_list)))
print(f'Sparsity: {S * 100}%')

Sparsity: 99.96113415774516%


- Một document có thể sử dụng một số lượng vocabulary cực kỳ nhỏ nhưng count vector tương ứng có `V` chiều (số vocabulary) thể hiện:
    + Tính nhất quán trong biểu diễn: Sau khi vectorize dựa trên từ điển vocab được xây dựng từ corpus, các vocab không xuất hiện trong document sẽ đánh bằng giá trị 0, điều này đảm bảo tính nhất quán khi biểu diễn ma trận thưa TF-IDF gồm `V` cột, trong đó các document được biểu diễn các count vector `V` chiều, sau đó sẽ đưa vào xử lý và trở thành vector thưa TF-IDF, mỗi vector này đại diện cho mỗi hàng của ma trận kể trên.
    + Tính toán độ tương đồng giữa các document: Khi truy vấn từ document trong corpus sử dụng công thức cosine similarity, 2 vector đại diện cho 2 document bất kỳ tương ứng phải có cùng số chiều `V`, cho phép tính tích vô hướng và độ tương đồng giữa 2 document.

- Inspect vocabulary

In [5]:
k = 20

max_df_index = np.argsort(df_vector)[::-1][:k]
print('20 terms phổ biến nhất theo document frequency')
for i in max_df_index:
    print(f'{vocabulary_list[i]}: {df_vector[i]}')

20 terms phổ biến nhất theo document frequency
the: 27870
and: 27385
to: 26646
of: 25999
a: 25753
in: 25042
for: 23556
is: 22673
with: 21323
on: 19936
that: 18042
this: 17548
are: 17487
as: 16355
at: 16233
from: 16183
be: 16025
it: 15474
you: 15361
by: 14963


In [6]:
max_idf_index = np.argsort(idf_vector)[::-1][:k]
print('20 terms có IDF cao nhất')
for i in max_idf_index:
    print(f'{vocabulary_list[i]}: {idf_vector[i]}')

20 terms có IDF cao nhất
 : 10.308952660644293
🤳:: 10.308952660644293
🤰:: 10.308952660644293
🤢🤢: 10.308952660644293
🤢.: 10.308952660644293
🤗🤗🤗: 10.308952660644293
🤖:: 10.308952660644293
🙏🏻: 10.308952660644293
🙌🏻👯❤: 10.308952660644293
🙌: 10.308952660644293
🙋!!: 10.308952660644293
ever: 10.308952660644293
000: 10.308952660644293
: 10.308952660644293
=: 10.308952660644293
: 10.308952660644293
remarkable: 10.308952660644293
!!!!!!: 10.308952660644293
!!!!some: 10.308952660644293
!!!).: 10.308952660644293


In [7]:
import random

index_doc = random.randint(0, N)
tf_idf_val_arr = tf_idf_matrix.tocsr()[index_doc]

values = tf_idf_val_arr.data
term_indices = tf_idf_val_arr.indices

max_tf_idf_index = np.argsort(values)[::-1][:k]
print(f'20 terms có TF-IDF cao nhất trong document {index_doc}')

for i in max_tf_idf_index:
    print(f'{vocabulary_list[term_indices[i]]}: {values[i]}')

20 terms có TF-IDF cao nhất trong document 15237
can't: 0.23030485866399655
spas: 0.1829292118862068
spa: 0.17066467818653666
dreammaker: 0.16234571119124871
eden: 0.16165078955538345
certified: 0.1345975189958909
blue: 0.1182234887821615
virginia,: 0.10042719729007821
distributor: 0.0995270647886617
america: 0.09763193835917791
dealer: 0.08569162535730251
klear: 0.0757150037801917
krystal: 0.0757150037801917
regale: 0.07252236513367073
furnaces,: 0.07025715196475907
help.: 0.06634847398759162
spas,: 0.06479930014932643
away.: 0.06386021809557196
'n: 0.06160666150280545
discuss: 0.0592016050143383


- Một term xuất hiện nhiều trong corpus chưa chắc có TF-IDF cao, chẳng hạn có thể tồn tại term xuất hiện gần như mọi document trong corpus, nhờ đó mà tăng giá trị DF của term lên, nhưng đồng thời giá trị IDF giảm đi, từ đó giá trị TF-IDF của term trong document có thể thấp và ngược lại, có thể có term chỉ xuất hiện trong rất ít document, dẫn đến việc kéo giá trị IDF đi lên, nhờ đó mà giá trị TF-IDF của term trong document sẽ cao hơn.
- Một term có IDF cao chưa chắc có giá trị TF-IDF cao trong mọi document. Chẳng hạn sẽ có term xuất hiện nhiều ở document này, dẫn đến TF của term trong document sẽ cao hơn, dẫn đến giá trị TF-IDF của term sẽ cao, nhưng xét trên document khác, cũng là term đang xét nhưng tần suất xuất hiện ít hơn, từ đó mà TF của nó sẽ thấp hơn, và giá trị TF-IDF sẽ giảm đi.

# Experiment 2: Preprocessing Ablation

- Mục tiêu: So sánh 3 tokenization preprocessing pipeline trên cùng corpus C4 (30K):
    + Pipeline A (Minimal): lowercase + tách khoảng trắng
    + Pipeline B (Normalized): lowercase + chuẩn hóa dấu câu + loại stopword
    + Pipeline C (Extended): chuẩn hóa khoảng trắng + BPE subword (BPE chỉ train trên tập train)
- Protocol: fit vocabulary trên train (80%); đo OOV và self-retrieval MRR@10 trên test (20%)

In [8]:
raw_corpus = corpus.copy()

In [9]:
# Chia tập corpus train/test theo tỷ lệ 8:2
from sklearn.model_selection import train_test_split

train_corpus, test_corpus = train_test_split(
    raw_corpus,
    test_size = 0.2,
    random_state = 42,
    shuffle = True
)

In [10]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer
from sklearn.preprocessing import normalize

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Pipeline A: Minimal - lowercase và tách khoảng trắng
def pipeline_A(document):
    # Viết thường các từ trong document
    document = document.lower()

    # Tách các token
    tokens = document.split()

    return tokens

# Pipeline B: Normalized - lowercase + chuẩn hóa dấu câu + loại stopword
def pipeline_B(document):
    document = document.lower()
    # Tách dấu câu khỏi từ
    document = re.sub(r"[^\w\s]+", " ", document, flags=re.UNICODE)
    document = re.sub(r"\s+", " ", document).strip()
    tokens = document.split()
    tokens = [token for token in tokens if token not in ENGLISH_STOP_WORDS]
    return tokens

# Pipeline C: Extended - Normalization + BPE subword tokenization
# Bước 1: Normalization
def normalize_C(document):
    document = document.lower()                         # Chuyển toàn bộ document về chữ thường
    document = re.sub(r"\s+", " ", document).strip()    # Gộp mọi chuỗi khoảng trắng liên tiếp (Tab, xuống dòng, dấu cách) thành một dấu cách duy nhất, rồi dùng strip() để xóa khoảng trắng thừa ở đầu và cuối câu
    return document

# Áp dụng thuật toán BPE cho subword tokenization
# BPE sẽ gộp các ký tự/subword xuất hiện thường xuyên thành một token mới và lặp lại quá trình này cho đến khi đạt kích thước vocab mong muốn
bpe_tokenizer = Tokenizer(
    BPE(unk_token="[UNK]")  # Token dự phòng cho các kỳ tự thực sự chưa từng thấy trong lúc train
)

# Trước khi áp dụng BPE, document được tách theo khoảng trắng và dấu câu thành các từ "thô". BPE chỉ gộp subword trong phạm vi của mỗi từ "thô".
bpe_tokenizer.pre_tokenizer = Whitespace()

bpe_trainer = BpeTrainer(
    vocab_size=30000,           # Giới hạn số lượng token tối đa trong từ điển cuối cùng
    min_frequency=2,            # Một cặp subword chỉ được gộp thành token mới nếu nó xuất hiện ít nhất 2 lần trong corpus
    special_tokens=["[UNK]"]    # Đảm bảo token đặc biệt này luôn có mặt trong từ điển, không bị BPE học đè
)

# Chuẩn hóa corpus dạng thô để huấn luyện
normalized_train_corpus = [
    normalize_C(document)
    for document in train_corpus
]

# Huấn luyện BPE
bpe_tokenizer.train_from_iterator(
    normalized_train_corpus,
    trainer=bpe_trainer
)

# Hàm xử lý pipeline
def pipeline_C(document):
    document = normalize_C(document)

    encoding = bpe_tokenizer.encode(document)

    return encoding.tokens

In [11]:
# Pipeline A
count_vectorizer_A = CountVectorizer(
    tokenizer = pipeline_A,
    preprocessor = None,
    token_pattern = None,
    lowercase = False
)

count_matrix_A = count_vectorizer_A.fit_transform(train_corpus)

# Pipeline B
count_vectorizer_B = CountVectorizer(
    tokenizer = pipeline_B,
    preprocessor = None,
    token_pattern = None,
    lowercase = False
)

count_matrix_B = count_vectorizer_B.fit_transform(train_corpus)

# Pipeline C
count_vectorizer_C = CountVectorizer(
    tokenizer = pipeline_C,
    preprocessor = None,
    token_pattern = None,
    lowercase = False
)

count_matrix_C = count_vectorizer_C.fit_transform(train_corpus)

In [12]:
from typing import Any


def count_TF_IDF_matrix(count_matrix, return_idf=False):
    # Ma trận TF — tránh chia 0 khi document rỗng
    document_lengths = np.asarray(count_matrix.sum(axis=1)).ravel().astype(float)
    safe_lengths = np.where(document_lengths > 0, document_lengths, 1.0)
    tf_matrix = count_matrix.multiply(1 / safe_lengths[:, None])

    # Vector IDF
    N = count_matrix.shape[0]
    df_vector = np.asarray((count_matrix > 0).sum(axis=0)).ravel()

    idf_vector = np.zeros(df_vector.shape[0], dtype=float)
    mask = df_vector > 0
    idf_vector[mask] = np.log(N / df_vector[mask])

    tf_idf_matrix = tf_matrix.multiply(idf_vector)
    return (tf_idf_matrix, idf_vector) if return_idf else tf_idf_matrix

# Tính trung bình token trong 1 document
def average_tokens_per_document(count_matrix):
    total_tokens_per_document = np.asarray(count_matrix.sum(axis = 1)).ravel()
    total_tokens = sum(total_tokens_per_document)
    total_documents = count_matrix.shape[0]
    return total_tokens / total_documents

# Tính độ thưa trong ma trận TF-IDF
def sparsity(tf_idf_matrix):
    N, V = tf_idf_matrix.shape
    S = 1 - (tf_idf_matrix.nnz / (N * V))
    return S

# Tính tỷ lệ OOV
def count_OOV_rate(test_corpus, vectorizer):
    train_vocab = set(vectorizer.vocabulary_.keys())
    analyzer = vectorizer.build_analyzer()

    total_tokens = 0
    oov_tokens = 0

    for document in test_corpus:
        tokens = analyzer(document)
        total_tokens += len(tokens)

        for token in tokens:
            if token not in train_vocab:
                oov_tokens += 1

    if total_tokens == 0:
        return 0.0

    return oov_tokens / total_tokens

# Đánh giá search performance bằng Mean Reciprocal Rank (MRR)
def mrr(queries, vectorizer, count_matrix, top_k=10):
    """
    Self-retrieval MRR@k, tính theo lô.
    queries: list[dict] - [{"query": str, "relevant_doc_id": int}, ...]
    vectorizer: CountVectorizer đã fit trên train
    count_matrix: count matrix của tập document dùng để search (test_corpus)
    """
    tf_idf_matrix, idf_vector = count_TF_IDF_matrix(count_matrix, return_idf=True)
    doc_matrix = normalize(tf_idf_matrix.tocsr())   # Chuẩn hóa ma trận TF-IDF trên tập train theo chuẩn 'l2', hỗ trợ phần tính toán cosine similarity

    query_texts = [item["query"] for item in queries]
    relevant_ids = np.array([item["relevant_doc_id"] for item in queries])

    # Vectorize tập query về dạng count matrix
    query_count = vectorizer.transform(query_texts)

    # Tính ma trận TF-IDF trên tập query
    query_lengths = np.asarray(query_count.sum(axis=1)).ravel().astype(float)
    safe_query_lengths = np.where(query_lengths > 0, query_lengths, 1.0)

    query_tfidf = query_count.multiply(1 / safe_query_lengths[:, None]).multiply(idf_vector)
    query_tfidf = normalize(query_tfidf.tocsr()).tolil()

    # Xử lý case độ dài câu truy vấn bằng 0 khi tính ma trận TF-IDF
    empty_mask = query_lengths == 0
    if np.any(empty_mask):
        query_tfidf[empty_mask] = 0
    query_tfidf = query_tfidf.tocsr()

    # Tính cosine similarity
    similarities = (query_tfidf @ doc_matrix.T).toarray()   # Ma trận similarity có kích thước (query.length x 30K)
    similarities[empty_mask] = -np.inf

    # Tính MRR
    ranked = np.argsort(-similarities, axis=1)[:, :top_k]   # Lọc ra k document có ranking similarity cao nhất theo từng query
    reciprocal_ranks = np.zeros(len(queries), dtype=float)
    for i, rel in enumerate(relevant_ids):
        if empty_mask[i]:
            continue
        matched = np.where(ranked[i] == rel)[0]
        if len(matched) > 0:
            reciprocal_ranks[i] = 1.0 / (matched[0] + 1)

    return float(reciprocal_ranks.mean())

def generate_queries_from_test(test_corpus, n_queries=300, random_state=42):
    """
    Self-retrieval: lấy câu đầu của document làm query, chính document đó là relevant.
    Chỉ lấy n_queries mẫu để ablation chạy kịp.
    """
    # Chọn ra các document ngẫu nhiên trên tập test
    rng = np.random.default_rng(random_state)
    indices = rng.choice(len(test_corpus), size=min(n_queries, len(test_corpus)), replace=False)

    # Tạo query từ các document
    queries = []
    for doc_id in indices:
        document = test_corpus[doc_id]
        query_text = document.split(".")[0].strip() #Lấy câu đầu tiên trong document
        if not query_text:
            query_text = document[:200] # Nếu rỗng, lấy 200 ký tự đầu trong document
        queries.append({"query": query_text, "relevant_doc_id": int(doc_id)})   # Add vào danh sách truy vấn
    return queries

In [13]:
vocabulary_size = [
    count_matrix_A.shape[1], 
    count_matrix_B.shape[1], 
    count_matrix_C.shape[1]
]
average_tokens_per_doc = [
    average_tokens_per_document(count_matrix_A),
    average_tokens_per_document(count_matrix_B),
    average_tokens_per_document(count_matrix_C)
]

tf_idf_matrix_A = count_TF_IDF_matrix(count_matrix_A)
tf_idf_matrix_B = count_TF_IDF_matrix(count_matrix_B)
tf_idf_matrix_C = count_TF_IDF_matrix(count_matrix_C)

matrix_sparsity = [
    sparsity(tf_idf_matrix_A) * 100,
    sparsity(tf_idf_matrix_B) * 100,
    sparsity(tf_idf_matrix_C) * 100
]

oov_rate = [
    count_OOV_rate(test_corpus, count_vectorizer_A) * 100,
    count_OOV_rate(test_corpus, count_vectorizer_B) * 100,
    count_OOV_rate(test_corpus, count_vectorizer_C) * 100
]

# Sinh bộ query dùng chung cho cả 3 pipeline
queries = generate_queries_from_test(test_corpus, n_queries=300, random_state=42)

# Ma trận đếm của test_corpus theo từng vectorizer
test_count_matrix_A = count_vectorizer_A.transform(test_corpus)
test_count_matrix_B = count_vectorizer_B.transform(test_corpus)
test_count_matrix_C = count_vectorizer_C.transform(test_corpus)

mrr_score = [
    mrr(queries, count_vectorizer_A, test_count_matrix_A, top_k=10),
    mrr(queries, count_vectorizer_B, test_count_matrix_B, top_k=10),
    mrr(queries, count_vectorizer_C, test_count_matrix_C, top_k=10),
]

- Bảng kết quả

In [14]:
import pandas as pd

df = pd.DataFrame(
    [vocabulary_size, average_tokens_per_doc, matrix_sparsity, oov_rate, mrr_score],
    columns = ['Pipeline A', 'Pipeline B', 'Pipeline C'],
    index = ['Vocabulary size', 'Average tokens/document', 'Matrix sparsity (%)', 'OOV rate (%)', 'Search performance (MRR@10)']
)

df.loc['Vocabulary size'] = df.loc['Vocabulary size'].astype(int)
df.loc['Average tokens/document'] = df.loc['Average tokens/document'].round(2)
df.loc['Matrix sparsity (%)'] = df.loc['Matrix sparsity (%)'].round(2)
df.loc['OOV rate (%)'] = df.loc['OOV rate (%)'].round(2)
df.loc['Search performance (MRR@10)'] = df.loc['Search performance (MRR@10)'].round(4)

df

,Pipeline A,Pipeline B,Pipeline C
Vocabulary size,403095.0000,167777.000,29670.000
Average tokens/document,361.6800,197.230,451.250
Matrix sparsity (%),99.9500,99.930,99.360
OOV rate (%),4.2600,3.600,0.030
Search performance (MRR@10),0.8712,0.887,0.896


1. Lowercasing giúp giảm bớt những token giống nhau nhưng tách biệt do phân biệt chữ hoa/thường. Ví dụ: Power/power
2. Stopword removal không phải lúc nào cũng cải thiện respresentation.
3. Việc loại punctuation có thể làm mất tính nhận diện của token. Ví dụ: số thập phân
4. Pipeline A tạo ra sparse matrix nhất
5. Pipeline C cho search tốt nhất
6. Search tốt hơn thì chưa chắc vocabulary nhỏ hơn.

# Application: Build a Document Search Engine

In [15]:
from sklearn.preprocessing import normalize as sk_normalize

# Chuẩn hóa (L2-norm) ma trận TF-IDF của corpus
doc_matrix = sk_normalize(tf_idf_matrix.tocsr())

print(f'Doc matrix shape: {doc_matrix.shape}')
print(f'Doc matrix format: {type(doc_matrix)}')


Doc matrix shape: (30000, 473388)
Doc matrix format: <class 'scipy.sparse._csr.csr_matrix'>


In [16]:
import numpy as np


def search(query, vectorizer, idf_vector, doc_matrix, corpus, top_k=5):
    """
    Hệ thống tìm kiếm document dựa trên TF-IDF + Cosine Similarity.

    Input:
        query      : câu truy vấn do người dùng nhập (raw text)
        vectorizer : CountVectorizer đã fit trên corpus
        idf_vector : vector IDF tính trên toàn bộ corpus, shape = (V,)
        doc_matrix : ma trận TF-IDF của corpus, đã được chuẩn hóa L2
                     (mỗi document là một vector đơn vị)
        corpus     : dữ liệu gốc (dùng để lấy đoạn preview hiển thị)
        top_k      : số lượng document trả về

    Output:
        list các tuple (rank, doc_id, similarity_score, document_preview),
        sắp xếp theo similarity giảm dần. Trả về [] nếu query không chứa
        bất kỳ term nào nằm trong vocabulary của corpus.
    """

    # Count vector của query
    query_count = vectorizer.transform([query]).toarray()[0]

    total_terms = query_count.sum()
    if total_terms == 0:
        return []

    # TF của query
    query_tf = query_count / total_terms

    # TF-IDF của query, dùng idf_vector đã tính trên toàn corpus
    query_tfidf = query_tf * idf_vector

    # Chuẩn hóa (L2) query vector để cosine similarity
    query_norm = np.linalg.norm(query_tfidf)
    if query_norm == 0:
        return []
    query_vector = query_tfidf / query_norm

    # Cosine similarity giữa query và toàn bộ corpus.
    scores = doc_matrix @ query_vector

    # Ranking theo similarity giảm dần, lấy top_k
    ranked_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for rank, doc_id in enumerate(ranked_indices, start=1):
        score = scores[doc_id]
        snippet = corpus[doc_id][:300].replace("\n", " ")
        results.append((rank, int(doc_id), float(score), snippet))

    return results


- Hàm hiển thị kết quả search và in bảng kết quả

In [17]:
def display_search_results(query, results):
    print(f"Query: {query}")
    print("-" * 80)

    if not results:
        print("No matching documents.")
        return

    for rank, doc_id, score, snippet in results:
        print(f"Rank: {rank}")
        print(f"Document ID: {doc_id}")
        print(f"Similarity: {score:.4f}")
        print(f"Preview: {snippet}")
        print("-" * 80)

In [18]:
import pandas as pd


def search_results_table(query, results):
    """
    Hiển thị kết quả search dưới dạng bảng
    """
    if not results:
        print(f'Query: {query}')
        print('No matching documents.')
        return pd.DataFrame(columns=['Rank', 'Document ID', 'Similarity', 'Document preview'])

    table = pd.DataFrame(
        [(rank, doc_id, round(score, 4), snippet) for rank, doc_id, score, snippet in results],
        columns=['Rank', 'Document ID', 'Similarity', 'Document preview']
    )
    return table

- Demo search engine trên các queries

In [27]:
queries = [
    "medical image classification",
    "transformer language model",
    "deep learning healthcare",
    "natural language processing",
    "computer vision",
    "data mining algorithm",
    "heart attack treatment"
]

all_results = {}

for query in queries:
    results = search(
        query,
        vectorizer,
        idf_vector,
        doc_matrix,
        corpus,
        top_k=5
    )

    all_results[query] = results
    display_search_results(query, results)

Query: medical image classification
--------------------------------------------------------------------------------
Rank: 1
Document ID: 27352
Similarity: 0.6188
Preview: The TESTID statement is effective only when you specify the TESTLIST or TESTLISTERR option in the PROC DISCRIM statement. When the DISCRIM procedure displays the classification results for the TESTDATA= data set, the TESTID variable (rather than the observation number) is displayed for each observat
--------------------------------------------------------------------------------
Rank: 2
Document ID: 13607
Similarity: 0.2036
Preview: Stunning Watch More Like Paint Storage Containers Paint Storage Containers - The image above with the title Stunning Watch More Like Paint Storage Containers Paint Storage Containers, is part of Paint Storage Containers picture gallery. Size for this image is 630 × 472, a part of Storage Containers 
--------------------------------------------------------------------------------
Rank: 3
D

In [28]:
combined_table = pd.concat(
    [search_results_table(q, r).assign(Query=q) for q, r in all_results.items()],
    ignore_index=True
)[['Query', 'Rank', 'Document ID', 'Similarity', 'Document preview']]

combined_table

,Query,Rank,Document ID,Similarity,Document preview
0,medical image classification,1,27352,0.6188,The TESTID statement is effective only when yo...
1,medical image classification,2,13607,0.2036,Stunning Watch More Like Paint Storage Contain...
2,medical image classification,3,10026,0.1855,Color classification of power input and output...
3,medical image classification,4,18949,0.1639,Newest Baby Car Seat Replacement Cover for Inf...
4,medical image classification,5,10368,0.1599,Published 04/20/2019 10:40:34 pm at 04/20/2019...
5,transformer language model,1,6326,0.3100,Label: butterfly twin comforter set. butterfly...
6,transformer language model,2,27936,0.2748,"hi, I am having problems with transformer / ci..."
7,transformer language model,3,11723,0.1665,Welcome to our Piro vocabulary page! Piro is a...
8,transformer language model,4,26951,0.1482,"Tinashe, cover star of the Autumn Winter 18 is..."
9,transformer language model,5,15853,0.1280,Why does my battery drain fast when I'm not us...


# Evaluation

In [29]:
# In ra candidate documents (top-K theo TF-IDF search ở Part G)
for query in queries:
    print(f"Query: {query}")
    print("-" * 80)
    for rank, doc_id, score, snippet in all_results[query]:
        print(f"  [{doc_id}] score={score:.4f} | {snippet}")
    print()

manual_relevance = {
    "medical image classification": [],
    "transformer language model": [],
    "deep learning healthcare": [],
    "natural language processing": [],
    "computer vision": [],
    "data mining algorithm": [3298, 27213],
    "heart attack treatment": [18006, 25228]
}

evaluation_set = [
    {"query": q, "relevant_doc_ids": manual_relevance[q]}
    for q in queries
]

for i, item in enumerate(evaluation_set, start=1):
    print(f"Query {i}: {item['query']}")
    print(f"Relevant: {item['relevant_doc_ids']}")
    print('-' * 80)

Query: medical image classification
--------------------------------------------------------------------------------
  [27352] score=0.6188 | The TESTID statement is effective only when you specify the TESTLIST or TESTLISTERR option in the PROC DISCRIM statement. When the DISCRIM procedure displays the classification results for the TESTDATA= data set, the TESTID variable (rather than the observation number) is displayed for each observat
  [13607] score=0.2036 | Stunning Watch More Like Paint Storage Containers Paint Storage Containers - The image above with the title Stunning Watch More Like Paint Storage Containers Paint Storage Containers, is part of Paint Storage Containers picture gallery. Size for this image is 630 × 472, a part of Storage Containers 
  [10026] score=0.1855 | Color classification of power input and output connector prevent misoperation of connecting sockets. P5.2 Indoor audiovisual production LED screen is the best option for event on stages, TV stations and con

- Tính Precision@K, Recall@K

In [30]:
def precision_at_k(retrieved_doc_ids, relevant_doc_ids, k=5):
    retrieved_k = retrieved_doc_ids[:k]
    n_relevant_retrieved = len(set(retrieved_k) & set(relevant_doc_ids))
    return n_relevant_retrieved / k


def recall_at_k(retrieved_doc_ids, relevant_doc_ids, k=5):
    if len(relevant_doc_ids) == 0:
        return 0.0
    retrieved_k = retrieved_doc_ids[:k]
    n_relevant_retrieved = len(set(retrieved_k) & set(relevant_doc_ids))
    return n_relevant_retrieved / len(relevant_doc_ids)


- Tính MRR

In [31]:
def reciprocal_rank(retrieved_doc_ids, relevant_doc_ids):
    for rank, doc_id in enumerate(retrieved_doc_ids, start=1):
        if doc_id in relevant_doc_ids:
            return 1.0 / rank
    return 0.0


def mean_reciprocal_rank(all_reciprocal_ranks):
    if len(all_reciprocal_ranks) == 0:
        return 0.0
    return sum(all_reciprocal_ranks) / len(all_reciprocal_ranks)



In [32]:
K = 5
SEARCH_TOP_K = 10  # lấy rộng hơn K để MRR có thể "bắt" được relevant doc nằm ngoài top-5

eval_records = []
all_reciprocal_ranks = []

for item in evaluation_set:
    query = item['query']
    relevant_doc_ids = item['relevant_doc_ids']

    results = search(query, vectorizer, idf_vector, doc_matrix, corpus, top_k=SEARCH_TOP_K)
    retrieved_doc_ids = [doc_id for (_, doc_id, _, _) in results]

    p_at_k = precision_at_k(retrieved_doc_ids, relevant_doc_ids, k=K)
    r_at_k = recall_at_k(retrieved_doc_ids, relevant_doc_ids, k=K)
    rr = reciprocal_rank(retrieved_doc_ids, relevant_doc_ids)
    all_reciprocal_ranks.append(rr)

    eval_records.append({
        'query': query,
        'relevant_doc_ids': relevant_doc_ids,
        'retrieved_doc_ids': retrieved_doc_ids,
        f'P@{K}': p_at_k,
        f'R@{K}': r_at_k,
        'RR': rr
    })

mrr_score_eval = mean_reciprocal_rank(all_reciprocal_ranks)

eval_df = pd.DataFrame(eval_records)
eval_df


,query,relevant_doc_ids,retrieved_doc_ids,P@5,R@5,RR
0,medical image classification,[],"[27352, 13607, 10026, 18949, 10368, 11534, 639...",0.0,0.0,0.0
1,transformer language model,[],"[6326, 27936, 11723, 26951, 15853, 6443, 9389,...",0.0,0.0,0.0
2,deep learning healthcare,[],"[26858, 19308, 3969, 24534, 26542, 12746, 1599...",0.0,0.0,0.0
3,natural language processing,[],"[11723, 16721, 6445, 9389, 6450, 13690, 8632, ...",0.0,0.0,0.0
4,computer vision,[],"[1244, 26714, 25950, 21081, 10961, 20115, 5497...",0.0,0.0,0.0
5,data mining algorithm,"[3298, 27213]","[4154, 3298, 10677, 28305, 27213, 11247, 12987...",0.4,1.0,0.5
6,heart attack treatment,"[18006, 25228]","[9048, 18006, 27200, 22261, 4778, 25228, 13602...",0.2,0.5,0.5


In [33]:
summary_df = pd.DataFrame({
    f'Precision@{K}': [eval_df[f'P@{K}'].mean()],
    f'Recall@{K}': [eval_df[f'R@{K}'].mean()],
    'MRR': [mrr_score_eval]
})

print('Kết quả đánh giá trung bình trên evaluation set:')
summary_df


Kết quả đánh giá trung bình trên evaluation set:


,Precision@5,Recall@5,MRR
0,0.085714,0.214286,0.142857


In [34]:
# Lưu kết quả retrieval/evaluation ra results.csv (deliverable của lab, mục 17)
results_export = eval_df.copy()
results_export.to_csv('results.csv', index=False)
print('Đã lưu results.csv')


Đã lưu results.csv
